# Retail Store Sales Drivers — Multilinear Regression
**Dataset:** Men's Fashion Stores, Netherlands | 400 stores | 1990  
**Objective:** Identify the primary drivers of annual sales using OLS regression with log-linear modeling.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/retail_sales_regression.ipynb)

---


## Setup — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("Libraries loaded.")

## Step 1 — Pick the Variables

**Target (y):** `log(tsales)` — annual sales log-transformed to fix right skew (raw skew: 2.39 → -0.69 after log)

**Predictors (X)** selected on business logic and correlation analysis:

| Variable | Description | Correlation with log(tsales) |
|---|---|---|
| `log_ssize` | Store floor space (m²) | **0.646** |
| `hoursw` | Total staff hours worked | **0.683** |
| `margin` | Gross profit margin | 0.365 |
| `npart` | Number of part-time workers | 0.304 |

**Dropped variables:** `nown` (r=0.172), `start` (r=0.224), `inv1`/`inv2` (not statistically significant), `hourspw` (correlated 0.81 with `hoursw` — multicollinearity risk)


In [ ]:
# Load dataset
data = pd.read_csv('Clothing.csv')

# Build working dataframe with transformed variables
df = pd.DataFrame({
    'log_tsales' : np.log(data['tsales']),
    'log_ssize'  : np.log(data['ssize']),
    'hoursw'     : data['hoursw'],
    'margin'     : data['margin'],
    'npart'      : data['npart'],
})

print(f"Dataset: {df.shape[0]} stores  |  {df.shape[1]} variables\n")
df.head(10)

## Step 2 — Summary Statistics

Descriptive statistics reveal the distribution and spread of each variable before modeling.


In [ ]:
df.describe().round(2)

## Step 3 — Correlation Matrix

Examines linear relationships between variables. Values close to ±1 signal a strong relationship — or a multicollinearity risk if between two predictors.


In [ ]:
print("Correlations with log(tsales):")
print(df.corr()['log_tsales'].sort_values(ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, center=0, cmap='coolwarm',
            ax=ax, fmt='.2f', linewidths=0.5)
ax.set_title('Correlation Matrix — Final Model Variables',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## Step 4 — Train / Test Split (80 / 20)

The data is split into a training set (model learns from this) and a test set (held out to grade the model on data it has never seen). The constant adds the intercept term required for OLS.


In [ ]:
X = sm.add_constant(df.drop('log_tsales', axis=1))
y = df['log_tsales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1502
)

print(f"Training set : {len(X_train)} stores  (80%)")
print(f"Test set     : {len(X_test)} stores  (20%)")

## Step 5 — Build the Model

OLS (Ordinary Least Squares) regression is fit on the training set. The target is `log(tsales)`, so coefficients represent **percentage change** in sales per unit change in each predictor.


In [ ]:
model = sm.OLS(y_train, X_train).fit()
print(model.summary())

## Step 6 — Multicollinearity Check (VIF)

**Variance Inflation Factor (VIF)** measures how much each predictor's variance is inflated by correlation with other predictors.

- VIF < 5 → acceptable  
- VIF 5–10 → monitor  
- VIF > 10 → concern, consider dropping


In [ ]:
vif = pd.DataFrame({
    'Feature' : X_train.columns,
    'VIF'     : [variance_inflation_factor(X_train.values, i)
                 for i in range(X_train.shape[1])]
}).set_index('Feature').round(2)

print(vif.to_string())
print("\nAll predictors below VIF threshold of 5 — no multicollinearity concern.")

## Step 7 — Residual Diagnostics

Two plots validate whether the OLS assumptions hold:

- **Residuals vs Fitted:** Points should scatter randomly around zero. A pattern or funnel shape signals a problem.
- **Q-Q Plot:** Points following the diagonal line confirm residuals are approximately normally distributed.


In [ ]:
residuals = model.resid
fitted    = model.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Diagnostics', fontsize=14, fontweight='bold', y=1.01)

# Residuals vs Fitted
sns.residplot(x=fitted, y=residuals, lowess=True, ax=axes[0],
              scatter_kws={'alpha': 0.45, 'color': 'steelblue'},
              line_kws={'color': 'red', 'linewidth': 1.8})
axes[0].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[0].set_title('Residuals vs. Fitted', fontweight='bold')
axes[0].set_xlabel('Fitted Values (log scale)')
axes[0].set_ylabel('Residuals')

# Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].get_lines()[0].set(color='steelblue', alpha=0.5, markersize=4)
axes[1].get_lines()[1].set(color='red', linewidth=1.8)
axes[1].set_title('Normal Q-Q Plot', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 8 — Accuracy Assessment

Predictions are made in log space, then converted back with `exp()` before computing error metrics in actual guilders.

| Metric | Meaning |
|---|---|
| **MAE** | Average absolute prediction error |
| **MAPE** | Average % error per store |
| **RMSE** | Like MAE but penalizes large misses more heavily |


In [ ]:
log_pred        = model.predict(X_test)
y_pred_guilders = np.exp(log_pred)
y_test_guilders = np.exp(y_test)

mae  = mean_absolute_error(y_test_guilders, y_pred_guilders)
mape = mean_absolute_percentage_error(y_test_guilders, y_pred_guilders)
rmse = root_mean_squared_error(y_test_guilders, y_pred_guilders)

print(f"R²   : {model.rsquared:.3f}   — model explains {model.rsquared*100:.1f}% of sales variance")
print(f"MAE  : {mae:>12,.0f}  guilders")
print(f"MAPE : {mape*100:>11.2f}%")
print(f"RMSE : {rmse:>12,.0f}  guilders")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test_guilders, y_pred_guilders,
           alpha=0.55, color='steelblue', edgecolors='white', linewidth=0.4, s=50)
max_val = max(y_test_guilders.max(), y_pred_guilders.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.8, label='Perfect Prediction')
ax.set_xlabel('Actual Annual Sales (guilders)', fontsize=11)
ax.set_ylabel('Predicted Annual Sales (guilders)', fontsize=11)
ax.set_title(f'Actual vs. Predicted Annual Sales — R² = {model.rsquared:.3f}  |  MAPE = {mape*100:.1f}%',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Results — Coefficient Interpretation

Since the target is `log(tsales)`, each coefficient represents an approximate **% change in sales**.

| Predictor | Coefficient | Business Meaning |
|---|---|---|
| `log_ssize` | +0.341 | A 1% increase in floor space → ~0.34% more sales |
| `hoursw` | +0.006 | Each additional staff-hour worked → ~0.6% more sales |
| `margin` | +0.019 | Each +1 point in gross margin → ~1.9% more sales |
| `npart` | +0.087 | Each additional part-time worker → ~8.7% more sales |

**All four predictors are statistically significant (p < 0.05).**

---

## Business Recommendations

**1. Staffing is the #1 lever**  
Part-time workers carry the highest coefficient (0.087). Optimizing flexible headcount is the fastest route to higher sales before committing capital.

**2. Floor space investment pays off**  
Store size is the second strongest driver. Expansion or layout optimization has a measurable, quantifiable impact on revenue.

**3. Protect gross margin**  
Margin is statistically significant. Aggressive discounting strategies that erode margin are likely counterproductive to total revenue.

**4. What the model can't see (~43% of variance unexplained)**  
Location quality, local competition, foot traffic, and product mix are absent from this dataset. These are the natural next variables to source for a more complete model.
